# M5.4 — Suggested Experimental Starting Points.

# 0. Nguyên tắc chung trước khi thử nghiệm

**Train / Dev / Test**
- Train: training
- Dev: experiment, model selection, error analysis
- Test: vẫn sealed cho đến khi pipeline cuối cùng được chốt

**Manual 50 không phải evaluation set**
- Bộ S01-S50 đến từ Train, do đó nó chỉ dùng để: sinh observation, sinh hypothesis, lấy ví dụ để giải thích, không được dùng performance trên manual 50 để chứng minh model tốt, vì model đã được train trên chính Train

**Khi so hai model phải giữ những thứ có thể giữ được giống nhau**
- cùng train
- cùng dev
- cùng metric
- cùng cách report
- training budget tương đương nếu khả thi

Không nhất thiết mọi hyper params phải giống nhau vì mỗi hệ thống có đặc điểm khác nhau

**Primary metric phải theo ViHOS chính thức**
- Ở M5.4 ta chưa tự phát minh metric, người 2 khi implement phải kiểm tra:
    - official ViHOS paper/repository dùng evaluation thế nào;
    - prediction được chuyển về span như thế nào;
    - precision / recall / F1 được tính ra sao.
- Sau đó dùng một evaluation pipeline thống nhất cho mọi experiment

# 1. H1 - Contextual Modeling Hypothesis

Câu hỏi cần trả lời: Model sử dụng contextual representation có thực sự tốt hơn một baseline chủ yếu dựa vào surface/lexical information trên ViHOS hay không?

Đây là các câu hỏi từ các case: S05, S13, S18, S20, S32, S44, S45 và hard clean S03, S25, S37, S39, S48.

Literature chính P1 — ViHOS — “ViHOS: Hate Speech Spans Detection for Vietnamese”, vì paper gốc cũng ghi nhận context là một nguồn lỗi.

**E1 - Suggested smallest experiment**

Model A - Non-PLM / lexical-surface baseline
- Gợi ý đầu tiên: CRF sequence labeler: không phải vì CRF là model tốt nhất, mà vì nó cho ta một baseline dễ hiểu (surface / token features + local sequence structure => BIO)
    - Ví dụ feature có thể gồm: token hiện tại, neigboring token, prefix/suffix, casing/punctuation, token shape, một vài lexical features đơn giản
    - Người 2 hoàn toàn có thể chọn baseline khác nếu họ có lý do: logistic/token classifier, BiLSTM, BiLSTM-CRF, một baseline khác...
    - Tại sao cần baseline này ? Nếu chỉ chạy PhoBERT rồi báo F1: ta không biết contextual pretraining mang lại gì so với một hệ thống đơn giản

Model B - Contextual PLM sequence labeler
- Gợi ý: PhoBERT + token classification head
    - Tại sao lại PhoBERT: Vietnamese PLM đã được dùng trong P1 - ViHOS, là contextual representation rõ ràng, tạo đối trọng hợp lý với lexical/surface baseline
    - Người 2 có thể thay bằng model khác nếu giải thích được

E1 phải trả về gì:
- Tối thiểu:

| Model | Precision | Recall | F1 |
|---|---:|---:|---:|
| Surface/non-PLM baseline | ... | ... | ... |
| Contextual PLM | ... | ... | ... |

- Nhưng bảng tổng chưa đủ, phải lấy một số Dev erros và hỏi: Model contextual sửa được lỗi nào và model contextual vẫn sai gì

# 2. H2 - Social-Media Domain Hypothesis

Câu hỏi: Pretraining chuyên cho Vietnamese social media có mang lại lợi ích so với Vietnamese/general-domain PLM hay không, đặc biệt ở noisy text?

Manual evidence:
- - ko;
- cmnr;
- mắc dạy;
- slang;
- abbreviation;
- teencode;
- noisy punctuation...

**E2 - Suggested smallest experiment**

Ta muốn so hai contextual models:
- Model B: General/Vietnamese PLM ví dụ PhoBERT
- Model C: Vietnamese social-media PLM ví dụ ViSoBERT

Đây không phải perfect causal experiment vì:
- tokenizer có thể khác
- pretraining corpus có thể khác
- architecture có thể khác

Cho nên kết quả chỉ là: model comparison compatible với H2 chứ không phải proof tuyệt đối rằng domain pretraining là nguyên nhân duy nhất

Những thứ cố gắng giữ giống:
- Train;
- Dev;
- downstream BIO task;
- evaluation code;
- report format;
- training budget tương đối hợp lý.

E2 phải trả về:

| Model | Domain | P | R | F1 |
|---|---|---:|---:|---:|
| Vietnamese/general PLM | general | ... | ... | ... |
| Social-media PLM | social media | ... | ... | ... |


- Người 2 nên xem Dev prediction failtures có dạng:
    - abbreviation;
    - slang;
    - teencode;
     spelling noise;
    - nonstandard punctuation.
- Không nhất thiết phải annotate toàn bộ Dev.

# 3. H3 - Boundary Difficulty Hypothesis

Ta không cần model D, không có lý do phải xây ngay: CRF decoder / span head / boundary model. Trước tiên ta chỉ hỏi: Boundary có thực sự là một loại lỗi lớn của các model mà người 2 vừa chạy không. Đây là hypothesis được sinh từ S02, S04, S24, S30, S31, S40 và được củng cố bởi: P1 — ViHOS — “ViHOS: Hate Speech Spans Detection for Vietnamese”.

**E3 - Boundary error audit**

Dùng predictions từ:
- baseline
- contextual PLM
- social-media PLM
- hoặc bất kỳ model nào đã chạy

Ta cần phân biệt các dạng lỗi:
- Type A - Missed span:
    - Gold: [B-T I-T], Pred [O O]: Model không tìm thấy offensive region
- Type B - Spurious span:
    - Gold: [O O O], Pred: [B-T I-T O]: False positive hoàn toàn.
- Type C - Under-span
    - Gold: [B-T I-T I-T], Pred: [B-T I-T O]: Model bắt được đúng vùng nhưng thiếu phần cuối.
- Type D - Over-span
    - Gold [O B-T I-T O], Pred: [B-T I-T I-T O]: Model lấy dư token
- Type E - Shifted Boundary
    - Gold và prediction overlap nhưng start/end lệch
- Type F - Merge/Split
    - Gold có hai span nhưng model merge thành một hoặc gold một span nhưng model chia đôi

H3 cần trả về:

| Error type | Count | Example |
|---|---:|---|
| Complete miss | ... | ... |
| Spurious | ... | ... |
| Under-span | ... | ... |
| Over-span | ... | ... |
| Shifted | ... | ... |
| Merge/split | ... | ... |

Nếu H3 được support:
- Có thể tự do đặt hypothesis mới, đây chính là evidence mới sinh ra hypo mới trong phần thực nghiệm